<h1 style="color: green;">Home task: metrics</h1>

About Dataset
The fish market dataset is a collection of data related to various species of fish and their characteristics. This dataset is designed for polynomial regression analysis and contains several columns with specific information.

## Load data

In [26]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)


In [27]:
df = pd.read_csv('/Users/user/Desktop/Camp2025/lesson_10/data/Fish.csv')
df


,Species,Weight,Length1,Length2,Length3,Height,Width
0,Bream,242.0,23.2,25.4,30.0,11.5200,4.0200
1,Bream,290.0,24.0,26.3,31.2,12.4800,4.3056
2,Bream,340.0,23.9,26.5,31.1,12.3778,4.6961
3,Bream,363.0,26.3,29.0,33.5,12.7300,4.4555
4,Bream,430.0,26.5,29.0,34.0,12.4440,5.1340
...,...,...,...,...,...,...,...
154,Smelt,12.2,11.5,12.2,13.4,2.0904,1.3936
155,Smelt,13.4,11.7,12.4,13.5,2.4300,1.2690
156,Smelt,12.2,12.1,13.0,13.8,2.2770,1.2558
157,Smelt,19.7,13.2,14.3,15.2,2.8728,2.0672


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Species  159 non-null    object 
 1   Weight   159 non-null    float64
 2   Length1  159 non-null    float64
 3   Length2  159 non-null    float64
 4   Length3  159 non-null    float64
 5   Height   159 non-null    float64
 6   Width    159 non-null    float64
dtypes: float64(6), object(1)
memory usage: 8.8+ KB


In [29]:
df.describe()

,Weight,Length1,Length2,Length3,Height,Width
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,398.326415,26.247170,28.415723,31.227044,8.970994,4.417486
std,357.978317,9.996441,10.716328,11.610246,4.286208,1.685804
min,0.000000,7.500000,8.400000,8.800000,1.728400,1.047600
25%,120.000000,19.050000,21.000000,23.150000,5.944800,3.385650
50%,273.000000,25.200000,27.300000,29.400000,7.786000,4.248500
75%,650.000000,32.700000,35.500000,39.650000,12.365900,5.584500
max,1650.000000,59.000000,63.400000,68.000000,18.957000,8.142000


## Regression

In [30]:
X = df.drop('Weight', axis=1)
y = df['Weight']
X = pd.get_dummies(X, columns=['Species'], drop_first=True)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [32]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

In [33]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)


In [34]:
def regression_metrics(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"\n--- {model_name} ---")
    print(f"MAE: {mae:.2f}")
    print(f"MSE: {mse:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.4f}")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

results_reg = {}
results_reg["Linear Regression"] = regression_metrics(y_test, lr_preds, "Linear Regression")
results_reg["Random Forest"] = regression_metrics(y_test, rf_preds, "Random Forest")



--- Linear Regression ---
MAE: 65.30
MSE: 7007.38
RMSE: 83.71
R²: 0.9507

--- Random Forest ---
MAE: 44.30
MSE: 4560.18
RMSE: 67.53
R²: 0.9679


## Classification

In [35]:
X = df[['Length1', 'Length2', 'Length3', 'Height', 'Width']]
y = df['Species']

In [36]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

In [38]:
print("\n--- Classification (Species) ---")
print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1-score: {f1:.3f}")

print("classification_report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))


--- Classification (Species) ---
Accuracy: 0.781
Precision: 0.805
Recall: 0.781
F1-score: 0.792
classification_report:

              precision    recall  f1-score   support

       Bream       1.00      1.00      1.00        10
      Parkki       1.00      1.00      1.00         1
       Perch       0.75      0.67      0.71         9
        Pike       1.00      1.00      1.00         3
       Roach       0.00      0.00      0.00         1
       Smelt       1.00      1.00      1.00         5
   Whitefish       0.00      0.00      0.00         3

    accuracy                           0.78        32
   macro avg       0.68      0.67      0.67        32
weighted avg       0.80      0.78      0.79        32



/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [41]:
confusion_matrix(y_test, y_pred)

array([[10,  0,  0,  0,  0,  0,  0],
       [ 0,  1,  0,  0,  0,  0,  0],
       [ 0,  0,  6,  0,  3,  0,  0],
       [ 0,  0,  0,  3,  0,  0,  0],
       [ 0,  0,  1,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  5,  0],
       [ 0,  0,  1,  0,  2,  0,  0]])

# Results

## Regression

* Linear Regression showed good prediction quality (R² ≈ 0.95), but had higher errors (MAE ≈ 65.3, RMSE ≈ 83.7).

* Random Forest Regressor showed better accuracy (R² ≈ 0.97) and lower errors (MAE ≈ 44.3, RMSE ≈ 67.5), indicating the model's ability to better capture nonlinear dependencies.

Metrics used:

- MAE: shows the mean absolute error. Convenient for interpretation, but does not take into account large deviations.

- MSE: amplifies the impact of large errors, but is less intuitive.

- RMSE: returns the error in the same units as the target variable.

- R²: shows how much of the variation in the target variable is explained by the model.


Random Forest Regressor showed better results on all metrics: lower MAE, MSE, RMSE and higher R².

Linear Regression is inferior, especially in RMSE, which indicates larger deviations.

## Classification

- Accuracy: proportion of correctly classified examples. Good for balanced classes.

- Precision: proportion of correct positive predictions. Important when false positives are critical.

- Recall: proportion of true positives found. Important when it is important not to miss any.

- F1-score: harmonic mean between precision and recall. Balances both indicators.

- Classification Report + Confusion Matrix: give a detailed picture for each class.

In classification problems with unbalanced classes, Accuracy can be misleading. F1-score and Recall are more reliable metrics, especially when it is important not to miss rare classes.

# Conclusion

- Metrics are crucial for model evaluation: different problems require different metrics.

- RMSE and R² are key for regression.

- F1-score, Recall, and Confusion Matrix are the most informative for classification, especially when there is class imbalance.

- The choice of metrics affects the interpretation of the model quality, so it is important to choose them according to the context of the problem.